### Inférence d'EuroBERT fine-tuné
Ce notebook permet d'inférer le modèle fine-tuné contenu dans le dossier `model_eurobert_political` sur l'ensemble des messages (`clean_data/flat_all_interactions`)

In [ ]:
!pip install transformers
!pip install torch
!pip install scikit-learn
!pip install matplotlib
!pip install numpy
!pip install seaborn
!pip install krippendorff

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from sklearn.model_selection import train_test_split
import torch.optim as optim
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, accuracy_score
)
import pandas as pd
import os
import io
import csv
import numpy as np
import re
import sys
import seaborn as sns
from tqdm import tqdm, trange
import matplotlib.pyplot as plt
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

In [ ]:
def clean_paragraph_text(text):
    """Decode HTML entities and remove paragraph tags from a text string.
    
    Returns the original value unchanged if it is NaN (missing).
    """
    if pd.isna(text):
        return text  # Preserve NaNs; they will be caught downstream if needed

    replacements = {
        "&eacute;": "é",
        "&agrave;": "à",
        "&egrave;": "è",
        "&Icirc;": "Î",
        "&ccedil;": "ç",
        "&acirc;": "â",
        "</P> <P>": " ",   # Merge adjacent paragraph blocks into a single string
    }

    for key, val in replacements.items():
        text = text.replace(key, val)

    return text

### Préparation du dataset à classifier

In [ ]:
# Récupération des messages à classifier
all_interactions = pd.read_csv("flat_all_interactions_revert.csv", delimiter=',')
print(f"Total texts to classify: {len(all_interactions):,}")

# Même nettoyage du texte que pendant l'entraînement
all_interactions['texte'] = all_interactions['texte'].apply(clean_paragraph_text)

In [ ]:
# Séparation du dataset en 20 chunks de taille égale
# Cela limite l'usage de la VRAM du GPU et permet des sauvegardes intermédiaires
N_CHUNKS = 20
parts = np.array_split(all_interactions, N_CHUNKS)
print(f"Chunk sizes: {[len(p) for p in parts]}")

### Chargement du modèle

In [ ]:
# Charger le modèle et le tokenizer pour l'inférence
MODEL_SAVE_PATH = "model_eurobert_political"
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_SAVE_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_SAVE_PATH)

In [ ]:
# Envoyer le modèle sur GPU si disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    print(f"Using GPU for inference: {torch.cuda.get_device_name(0)}")
else:
    print("GPU not available. Using CPU — inference will be slower.")

model.to(device)

### Inférence

In [ ]:
INFERENCE_BATCH_SIZE = 256 # dépend de la VRAM du GPU disponible

for i in range(1, N_CHUNKS + 1):
    print(f"\n Chunk {i}/{N_CHUNKS}")
    chunk_text = parts[i - 1].copy()

    encodings = tokenizer(
        chunk_text.ravel().tolist(),
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors='pt'
    )
    dataset    = TensorDataset(encodings['input_ids'], encodings['attention_mask'])
    dataloader = DataLoader(dataset, batch_size=INFERENCE_BATCH_SIZE)

    model.eval()
    all_preds = []

    for batch in tqdm(dataloader):
        input_ids_b, attention_mask_b = tuple(t.to(device) for t in batch)
        with torch.no_grad():
            outputs = model(input_ids_b, attention_mask=attention_mask_b)
            logits  = outputs.logits               # shape: (batch_size, 2)
            preds   = torch.argmax(logits, dim=1)  # shape: (batch_size,)
            all_preds.extend(preds.cpu().numpy())

    df = pd.DataFrame(chunk_text, columns=['texte'])
    df['political'] = all_preds
    df_positive = df[df['political'] == 1].copy()

    out_path = f'output/data_political_{i}.csv'
    df_positive.to_csv(out_path, index=False)

Les sauvegardes temporaires sont placés dans le sous-dossier `output`.

### Fusion finale

In [ ]:
list_outputs = os.listdir('output')
first_file = list_outputs.pop(0)

df = pd.read_csv('output/'+first_file)
for file in list_outputs:
    temp = pd.read_csv(file)
    df = pd.concat([df, temp])
df = df.drop_duplicates()
df = df.rename(columns={'texte':'text'})
df = df.drop(columns=['political'])
df.to_csv('../../clean_data/flat_political_interactions.csv', index=False)
print(len(df))